# RetinaVessel AI — controlled training v2

This notebook repeats all three seed-42 development experiments under one declared policy: a maximum of 30 epochs with patience-5 early stopping. It writes exact-resume checkpoints after every epoch and packages the complete development artifacts.

**Protocol boundary:** this notebook uses only the frozen FIVES training/validation manifest. It does not evaluate, inspect, or tune against the official FIVES test set.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/simlexx-k/retinavessel-ai.git"
EXPECTED_COMMIT = "0c40aa665be3a247372062eaa46a33f578cf7b0f"
REPO = Path("/kaggle/working/retinavessel-ai-v2-code")

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)

subprocess.run(
    ["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")],
    check=True,
)
os.chdir(REPO)
actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
assert actual_commit == EXPECTED_COMMIT
print("Pinned training code:", actual_commit)
print("Working directory:", Path.cwd())

In [ ]:
import json
import torch

sys.path.insert(0, str(REPO / "src"))
from retinavessel.data import find_fives_root, load_manifest

DATA_MOUNT = Path("/kaggle/input/datasets/simonkosgei/fives-retinal-dataset")
SPLIT_MANIFEST = REPO / "splits/fives_seed42.json"
assert DATA_MOUNT.exists(), f"Missing Kaggle dataset: {DATA_MOUNT}"
assert torch.cuda.is_available(), "Enable a GPU accelerator before running v2"

FIVES_ROOT = find_fives_root(DATA_MOUNT)
official_train = load_manifest(FIVES_ROOT, "train")
official_test = load_manifest(FIVES_ROOT, "test")
split = json.loads(SPLIT_MANIFEST.read_text())
assert len(official_train) == 600
assert len(official_test) == 200
assert len(split["train"]) == 504
assert len(split["validation"]) == 89
assert len(split["excluded"]) == 7

print("PyTorch:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))
print("FIVES root:", FIVES_ROOT)
print("Frozen development split: 504 train / 89 validation / 7 excluded")
print("Official test count verified but not evaluated: 200")
subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv",
], check=True)

In [ ]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = str(REPO / "src")
subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO,
    env=test_environment,
    check=True,
)

## Controlled experiment configuration

All models use the same frozen split, seed, patch sampling budget, batch size, optimizer defaults, maximum epoch count, and early-stopping patience. Existing v1 checkpoints are intentionally not reused because they lack optimizer and random-number-generator state.

In [ ]:
import shlex
import time

MAX_EPOCHS = 30
PATIENCE = 5
SEED = 42
RUN_TIMES = {}

EXPERIMENTS = [
    {"name": "unet_seed42_es", "model": "unet", "degradation": False},
    {"name": "lite_unet_seed42_es", "model": "lite_unet", "degradation": False},
    {"name": "lite_robust_seed42_es", "model": "lite_unet", "degradation": True},
]

def run_experiment(spec):
    output = REPO / "outputs" / spec["name"]
    last_checkpoint = output / "last.pt"
    command = [
        sys.executable, "-m", "retinavessel.train",
        "--data", str(DATA_MOUNT),
        "--model", spec["model"],
        "--epochs", str(MAX_EPOCHS),
        "--patience", str(PATIENCE),
        "--batch-size", "2",
        "--patch-size", "384",
        "--patches-per-image", "2",
        "--split-manifest", str(SPLIT_MANIFEST),
        "--seed", str(SEED),
        "--device", "cuda",
        "--out", str(output),
    ]
    if spec["degradation"]:
        command.append("--degradation")

    if last_checkpoint.exists():
        state = torch.load(last_checkpoint, map_location="cpu", weights_only=False)
        complete = (
            int(state["epoch"]) >= MAX_EPOCHS
            or int(state["epochs_without_improvement"]) >= PATIENCE
        )
        if complete:
            print(f"{spec['name']} already complete at epoch {state['epoch']}; skipping")
            return
        command.extend(["--resume", str(last_checkpoint)])

    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(REPO / "src")
    print("Running:", shlex.join(command))
    started = time.perf_counter()
    subprocess.run(command, cwd=REPO, env=environment, check=True)
    RUN_TIMES[spec["name"]] = time.perf_counter() - started
    for required in ("best.pt", "last.pt", "history.json", "split_manifest.json"):
        assert (output / required).is_file(), f"Missing {output / required}"

## E1 — standard U-Net

In [ ]:
run_experiment(EXPERIMENTS[0])

## E2 — lightweight U-Net

In [ ]:
run_experiment(EXPERIMENTS[1])

## E3 — lightweight U-Net with training-only degradation

In [ ]:
run_experiment(EXPERIMENTS[2])

## Development-set summary and provenance

In [ ]:
import hashlib
import pandas as pd

from retinavessel.model import UNet, model_params

rows = []
for spec in EXPERIMENTS:
    output = REPO / "outputs" / spec["name"]
    history = json.loads((output / "history.json").read_text())
    checkpoint = torch.load(output / "best.pt", map_location="cpu", weights_only=True)
    network = UNet(
        base=int(checkpoint["base"]),
        lite=checkpoint["architecture"] == "lite_unet",
    )
    rows.append({
        "experiment": spec["name"],
        "model": checkpoint["architecture"],
        "degradation_training": spec["degradation"],
        "parameters": model_params(network),
        "completed_epochs": history[-1]["epoch"],
        "best_epoch": checkpoint["epoch"],
        "best_validation_dice": checkpoint["validation_dice"],
        "wall_minutes_this_notebook": round(RUN_TIMES.get(spec["name"], float("nan")) / 60, 2),
    })

summary = pd.DataFrame(rows).sort_values("best_validation_dice", ascending=False)
display(summary)

SUMMARY_DIR = REPO / "outputs" / "v2_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(SUMMARY_DIR / "validation_summary.csv", index=False)
run_manifest = {
    "training_code_commit": EXPECTED_COMMIT,
    "dataset_mount": str(DATA_MOUNT),
    "resolved_fives_root": str(FIVES_ROOT),
    "split_manifest_sha256": hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest(),
    "pytorch_version": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0),
    "maximum_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "seed": SEED,
    "official_test_evaluated": False,
    "results": json.loads(summary.to_json(orient="records")),
}
(SUMMARY_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
print("Summary written to", SUMMARY_DIR)

In [ ]:
ARCHIVE = Path("/kaggle/working/retinavessel_v2_seed42.zip")
if ARCHIVE.exists():
    ARCHIVE.unlink()
subprocess.run(
    [
        "zip", "-r", str(ARCHIVE),
        "outputs/unet_seed42_es",
        "outputs/lite_unet_seed42_es",
        "outputs/lite_robust_seed42_es",
        "outputs/v2_summary",
    ],
    cwd=REPO,
    check=True,
)
print(f"Artifact archive: {ARCHIVE} ({ARCHIVE.stat().st_size / 1024**2:.1f} MiB)")
print("Use Kaggle Save Version / Save & Run All, then download this archive.")

## Stop here

Do not run the official FIVES test evaluation from this notebook. The next stage is validation-only robustness, topology, latency, and peak-memory analysis, followed by multi-seed replication. The official test remains sealed until those decisions are frozen.